# PDF (2026년 최신 권장 사용법)

[Portable Document Format (PDF)](https://en.wikipedia.org/wiki/PDF)는 ISO 32000 으로 표준화된 파일 형식으로, Adobe 가 1992년에 문서를 제시하기 위해 개발했으며 응용 소프트웨어·하드웨어·운영체제와 독립적인 방식으로 텍스트 서식과 이미지를 담습니다.

이 가이드는 `PDF` 문서를 LangChain `Document` 형식으로 로드하는 방법을 다룹니다. PDF 파서는 간단한 저수준 텍스트 추출기부터 OCR·레이아웃 분석을 수행하는 고급 파서까지 다양하며, 올바른 선택은 애플리케이션에 따라 달라집니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `PyPDFLoader` | `pypdf` 직접 사용 → `Document` |
| `PyPDFLoader(..., extract_images=True)` + `rapidocr-onnxruntime` | `rapidocr` 패키지 직접 사용 (community 에서도 `rapidocr-onnxruntime` → `rapidocr` 로 교체됨) |
| `PyMuPDFLoader` | `pymupdf` 직접 사용, 또는 전용 패키지 **`langchain-pymupdf4llm`** 의 `PyMuPDF4LLMLoader` (Markdown 출력) |
| `UnstructuredPDFLoader(mode="elements")` | **`langchain-unstructured`** 의 `UnstructuredLoader` (기본이 요소 단위) |
| `PyPDFium2Loader` | `pypdfium2` 직접 사용 |
| `PDFMinerLoader`, `PDFMinerPDFasHTMLLoader` | `pdfminer.six` 직접 사용 |
| `PyPDFDirectoryLoader` | `pathlib.glob` + 위 로더 |
| `PDFPlumberLoader` | `pdfplumber` 직접 사용 |
| (신규) | **`langchain-docling`** 의 `DoclingLoader` |

**참고**
- [LangChain Document loader integrations (PDFs)](https://docs.langchain.com/oss/python/integrations/document_loaders)
- [PyMuPDF4LLMLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/pymupdf4llm)

## AutoRAG 팀에서의 PDF 실험 (책 원문 유지)

AutoRAG 에서 진행한 실험(2024년)을 토대로 작성한 순위표입니다. 아래 숫자는 등수를 나타냅니다. (The lower, the better)

| | PDFMiner | PDFPlumber | PyPDFium2 | PyMuPDF | PyPDF2 |
|----------|:---------:|:----------:|:---------:|:-------:|:-----:|
| Medical  | 1         | 2          | 3         | 4       | 5     |
| Law      | 3         | 1          | 1         | 3       | 5     |
| Finance  | 1         | 2          | 2         | 4       | 5     |
| Public   | 1         | 1          | 1         | 4       | 5     |
| Sum      | 5         | 5          | 7         | 15      | 20    |

출처: [AutoRAG 블로그](https://velog.io/@autorag/PDF-%ED%95%9C%EA%B8%80-%ED%85%8D%EC%8A%A4%ED%8A%B8-%EC%B6%94%EC%B6%9C-%EC%8B%A4%ED%97%98#%EC%B4%9D%ED%8F%89)

> 참고: `PyPDF2` 는 오래전에 `pypdf` 로 통합되었고, 이후 각 라이브러리도 버전이 크게 올랐습니다. 현재 버전 기준으로 다시 비교해 보는 것을 권장합니다.

In [ ]:
# 설치 (필요한 것만 골라 설치하세요)
# !pip install -qU langchain-core langchain-text-splitters python-dotenv
# !pip install -qU pypdf pymupdf pypdfium2 pdfminer.six pdfplumber beautifulsoup4

In [ ]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

## 실습에 활용한 문서

소프트웨어정책연구소(SPRi) - 2023년 12월호

- 저자: 유재흥(AI정책연구실 책임연구원), 이지수(AI정책연구실 위촉연구원)
- 링크: https://spri.kr/posts/view/23669
- 파일명: `SPRI_AI_Brief_2023년12월호_F.pdf`

**참고**: 위의 파일은 `data` 폴더 내에 다운로드 받으세요

In [ ]:
FILE_PATH = "./data/SPRI_AI_Brief_2023년12월호_F.pdf"

In [ ]:
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print("\n[examples]")
        max_key_length = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f"{k:<{max_key_length}} : {v}")

## 공통: 페이지 추출 함수 → Document 로더

community 로더들은 결국 "파서 라이브러리로 페이지 텍스트를 뽑고 → `Document` 로 감싸는" 일을 했습니다.

아래 `PagePDFLoader` 는 **페이지별 (텍스트, 메타데이터)를 yield 하는 함수**만 넘기면 어떤 파서든 LangChain 로더로 만들어 줍니다. 이후 섹션에서는 파서별 추출 함수만 작성합니다.

In [ ]:
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document

# (page_text, extra_metadata) 를 페이지마다 yield 하는 함수 타입
PageExtractor = Callable[[Path], Iterable[tuple[str, dict[str, Any]]]]


class PagePDFLoader(BaseLoader):
    """페이지 추출 함수를 받아 페이지 단위 Document 를 만드는 범용 PDF 로더"""

    def __init__(self, file_path: str | Path, extractor: PageExtractor, parser_name: str) -> None:
        self.file_path = Path(file_path)
        self.extractor = extractor
        self.parser_name = parser_name

    def lazy_load(self) -> Iterator[Document]:
        for page_number, (text, extra) in enumerate(self.extractor(self.file_path)):
            yield Document(
                page_content=text,
                metadata={
                    "source": str(self.file_path),
                    "page": page_number,  # 0부터 시작
                    "parser": self.parser_name,
                    **extra,
                },
            )

## PyPDF

`pypdf` 를 사용하여 PDF 를 페이지 단위 문서 배열로 로드합니다. 각 문서는 `page` 번호와 함께 페이지 내용 및 메타데이터를 포함합니다.

In [ ]:
# 설치
# !pip install -qU pypdf

In [ ]:
from pypdf import PdfReader


def pypdf_pages(path: Path):
    reader = PdfReader(path)
    # PDF 문서 정보(/Title, /Author ...) 를 메타데이터로 정리
    doc_info = {k.lstrip("/"): str(v) for k, v in (reader.metadata or {}).items()}
    total_pages = len(reader.pages)
    for page in reader.pages:
        # extraction_mode="layout" 으로 바꾸면 원본 배치를 좀 더 보존합니다.
        yield page.extract_text() or "", {"total_pages": total_pages, **doc_info}


loader = PagePDFLoader(FILE_PATH, pypdf_pages, "pypdf")

# 문서 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[10].page_content[:300])

In [ ]:
# 메타데이터 출력
show_metadata(docs)

### PyPDF (+ OCR)

일부 PDF 에는 스캔된 문서나 그림 속 텍스트 이미지가 포함되어 있습니다.

책에서는 `rapidocr-onnxruntime` 을 사용했지만, 현재는 후속 패키지인 **`rapidocr`** (+ `onnxruntime`) 를 사용합니다.
여기서는 `pypdf` 로 페이지 안의 이미지를 꺼내 OCR 결과를 본문 뒤에 덧붙입니다.

In [ ]:
# 설치
# !pip install -qU rapidocr onnxruntime

In [ ]:
import tempfile
import urllib.request

from rapidocr import RapidOCR

ocr_engine = RapidOCR()


def pypdf_pages_with_ocr(path: Path):
    reader = PdfReader(path)
    total_pages = len(reader.pages)
    for page in reader.pages:
        text = page.extract_text() or ""
        ocr_texts = []
        for image in page.images:  # 페이지에 포함된 이미지들
            result = ocr_engine(image.data)  # 이미지 bytes 를 바로 전달
            if result.txts:
                ocr_texts.append("\n".join(result.txts))
        if ocr_texts:
            text += "\n\n[이미지 OCR]\n" + "\n".join(ocr_texts)
        yield text, {"total_pages": total_pages, "ocr_images": len(ocr_texts)}


# 책과 동일한 arXiv 논문을 임시 폴더에 내려받아 사용 (URL 을 직접 받던 기능 대체)
pdf_url = "https://arxiv.org/pdf/2103.15348.pdf"
tmp_path = Path(tempfile.gettempdir()) / "2103.15348.pdf"
urllib.request.urlretrieve(pdf_url, tmp_path)

loader = PagePDFLoader(tmp_path, pypdf_pages_with_ocr, "pypdf+rapidocr")

# PDF 페이지 로드
docs = loader.load()

# 페이지 내용 접근
print(docs[4].page_content[:300])

In [ ]:
show_metadata(docs)

## PyMuPDF

**PyMuPDF** 는 속도 최적화가 되어 있으며, PDF 및 해당 페이지에 대한 자세한 메타데이터를 제공합니다. (패키지 import 이름은 이제 `fitz` 가 아닌 **`pymupdf`** 가 권장됩니다.)

> ⚖️ PyMuPDF 계열은 **AGPL/상용 라이선스**입니다. 사내·상용 서비스에 쓰기 전 라이선스를 확인하세요.

In [ ]:
# 설치
# !pip install -qU pymupdf

In [ ]:
import pymupdf


def pymupdf_pages(path: Path):
    with pymupdf.open(path) as pdf:
        doc_info = {k: v for k, v in pdf.metadata.items() if v}
        for page in pdf:
            yield page.get_text(), {"total_pages": pdf.page_count, **doc_info}


loader = PagePDFLoader(FILE_PATH, pymupdf_pages, "pymupdf")

# 문서 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[10].page_content[:300])

In [ ]:
show_metadata(docs)

### PyMuPDF4LLM (전용 통합 패키지 · Markdown 출력)

LangChain 공식 통합 목록에 있는 **`langchain-pymupdf4llm`** 패키지입니다. 제목·표·목록 구조를 **Markdown** 으로 보존하므로 RAG 청킹에 유리합니다.

In [ ]:
# 설치
# !pip install -qU langchain-pymupdf4llm

In [ ]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

loader = PyMuPDF4LLMLoader(
    FILE_PATH,
    mode="page",  # "single"(문서 전체 1개) 또는 "page"(페이지 단위)
    # table_strategy="lines",  # 표 추출 전략: "lines_strict"(기본) / "lines" / "text"
)

docs = loader.load()
print(docs[10].page_content[:500])

In [ ]:
show_metadata(docs)

## Unstructured

[Unstructured](https://docs.unstructured.io/)는 PDF, HTML, Word 등 비구조화/반구조화 문서를 다루는 공통 인터페이스를 제공합니다.

책의 `UnstructuredPDFLoader` 는 deprecated 되었고, 전용 패키지 **`langchain-unstructured`** 의 **`UnstructuredLoader`** 로 대체되었습니다.

- 파일 형식을 자동 감지합니다. (PDF 전용 클래스가 따로 없습니다)
- `mode` 인자가 없으며 **항상 요소(element) 단위**로 `Document` 를 반환합니다. (책의 `mode="elements"` 와 같은 결과)
- 요소를 묶고 싶으면 `chunking_strategy="by_title"` 등을 쓰거나, 직접 합칩니다.
- 로컬 파싱은 `unstructured[pdf]`, API 파싱은 `unstructured-client` + `partition_via_api=True` 를 사용합니다.

In [ ]:
# 설치 (로컬 파싱)
# !pip install -qU langchain-unstructured "unstructured[pdf]"

In [ ]:
from langchain_unstructured import UnstructuredLoader

# UnstructuredLoader 인스턴스 생성
loader = UnstructuredLoader(
    FILE_PATH,
    strategy="fast",  # "fast" | "hi_res"(레이아웃 모델 + OCR) | "ocr_only" | "auto"
    # languages=["kor", "eng"],  # hi_res / OCR 사용 시 언어 지정
)

# 데이터 로드 (요소 단위)
docs = loader.load()

# 첫 번째 요소의 내용 출력
print(docs[0].page_content)

책의 기본 모드(요소들을 하나로 합친 단일 문서)가 필요하면 직접 합치면 됩니다.

In [ ]:
single_doc = Document(
    page_content="\n\n".join(d.page_content for d in docs),
    metadata={"source": FILE_PATH},
)
print(single_doc.page_content[:300])

이 특정 문서에 대한 전체 요소 유형 집합을 확인해 봅니다.

In [ ]:
set(doc.metadata["category"] for doc in docs)  # 데이터 카테고리 추출

In [ ]:
show_metadata(docs)

## PyPDFium2

In [ ]:
# 설치
# !pip install -qU pypdfium2

In [ ]:
import pypdfium2 as pdfium


def pypdfium2_pages(path: Path):
    pdf = pdfium.PdfDocument(path)
    try:
        total_pages = len(pdf)
        for page in pdf:
            textpage = page.get_textpage()
            text = textpage.get_text_range()
            textpage.close()
            page.close()
            yield text, {"total_pages": total_pages}
    finally:
        pdf.close()


loader = PagePDFLoader(FILE_PATH, pypdfium2_pages, "pypdfium2")

# 데이터 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[10].page_content[:300])

In [ ]:
show_metadata(docs)

## PDFMiner

`pdfminer.six` 를 직접 사용합니다. 책의 `PDFMinerLoader` 는 기본적으로 문서 전체를 1개로 반환했지만, 여기서는 페이지 단위로 추출합니다.

In [ ]:
# 설치
# !pip install -qU pdfminer.six

In [ ]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer


def pdfminer_pages(path: Path):
    for page_layout in extract_pages(path):
        text = "".join(
            element.get_text()
            for element in page_layout
            if isinstance(element, LTTextContainer)
        )
        yield text, {}


loader = PagePDFLoader(FILE_PATH, pdfminer_pages, "pdfminer")

# 데이터 로드
docs = loader.load()

# 문서의 내용 출력
print(docs[0].page_content[:300])

In [ ]:
show_metadata(docs)

**PDFMiner** 를 사용하여 HTML 텍스트 생성

출력된 HTML 을 `BeautifulSoup` 으로 파싱하면 글꼴 크기, 페이지 번호, 헤더/푸터 등 보다 풍부한 정보를 얻을 수 있어, 텍스트를 의미 단위 섹션으로 나누는 데 도움이 됩니다.

In [ ]:
from io import StringIO

from pdfminer.high_level import extract_text_to_fp
from pdfminer.layout import LAParams

# PDF → HTML 문자열 (codec="" 으로 지정하면 str 로 기록됩니다)
html_buffer = StringIO()
with open(FILE_PATH, "rb") as f:
    extract_text_to_fp(f, html_buffer, codec="", laparams=LAParams(), output_type="html")

html_doc = Document(
    page_content=html_buffer.getvalue(),
    metadata={"source": FILE_PATH, "parser": "pdfminer-html"},
)
docs = [html_doc]

# 문서의 내용 출력
print(docs[0].page_content[:300])

In [ ]:
show_metadata(docs)

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(docs[0].page_content, "html.parser")  # HTML 파서 초기화
content = soup.find_all("div")  # 모든 div 태그 검색

In [ ]:
import re

cur_fs = None
cur_text = ""
snippets = []  # 동일한 글꼴 크기의 모든 스니펫 수집
for c in content:
    sp = c.find("span")
    if not sp:
        continue
    st = sp.get("style")
    if not st:
        continue
    fs = re.findall(r"font-size:(\d+)px", st)  # raw string 사용 (Python 3.12+ 경고 방지)
    if not fs:
        continue
    fs = int(fs[0])
    if not cur_fs:
        cur_fs = fs
    if fs == cur_fs:
        cur_text += c.text
    else:
        snippets.append((cur_text, cur_fs))
        cur_fs = fs
        cur_text = c.text
snippets.append((cur_text, cur_fs))
# 중복 스니펫 제거 전략 추가 가능 (PDF 의 헤더/푸터가 여러 페이지에 반복되므로 중복 정보로 간주 가능)

In [ ]:
cur_idx = -1
semantic_snippets = []
# 제목 가정: 높은 글꼴 크기
for s in snippets:
    # 새 제목 판별: 현재 스니펫 글꼴 > 이전 제목 글꼴
    if (
        not semantic_snippets
        or s[1] > semantic_snippets[cur_idx].metadata["heading_font"]
    ):
        metadata = {"heading": s[0], "content_font": 0, "heading_font": s[1]}
        metadata.update(docs[0].metadata)
        semantic_snippets.append(Document(page_content="", metadata=metadata))
        cur_idx += 1
        continue

    # 동일 섹션 내용 판별: 현재 스니펫 글꼴 <= 이전 내용 글꼴
    if (
        not semantic_snippets[cur_idx].metadata["content_font"]
        or s[1] <= semantic_snippets[cur_idx].metadata["content_font"]
    ):
        semantic_snippets[cur_idx].page_content += s[0]
        semantic_snippets[cur_idx].metadata["content_font"] = max(
            s[1], semantic_snippets[cur_idx].metadata["content_font"]
        )
        continue

    # 새 섹션 생성 조건: 현재 스니펫 글꼴 > 이전 내용 글꼴, 이전 제목 글꼴 미만
    metadata = {"heading": s[0], "content_font": 0, "heading_font": s[1]}
    metadata.update(docs[0].metadata)
    semantic_snippets.append(Document(page_content="", metadata=metadata))
    cur_idx += 1

print(semantic_snippets[4])

> 💡 **요즘 방식:** 글꼴 크기로 제목을 추정하는 대신, Markdown 을 출력하는 파서(`PyMuPDF4LLMLoader`, `DoclingLoader`, `UpstageDocumentParseLoader(output_format="markdown")` 등)로 제목 구조를 얻은 뒤 `MarkdownHeaderTextSplitter` 로 섹션 단위 분할하는 방법이 간단합니다.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_docs = PyMuPDF4LLMLoader(FILE_PATH, mode="single").load()

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
    strip_headers=False,
)
sections = header_splitter.split_text(md_docs[0].page_content)
print(len(sections))
sections[4] if len(sections) > 4 else sections[-1]

## PDF 디렉토리

디렉토리에서 PDF 를 로드합니다. (`PyPDFDirectoryLoader` 대체: `pathlib` + 위에서 만든 로더)

In [ ]:
from itertools import chain

pdf_paths = sorted(Path("data/").glob("*.pdf"))  # 하위 폴더까지: rglob("*.pdf")

docs = list(
    chain.from_iterable(
        PagePDFLoader(p, pypdf_pages, "pypdf").lazy_load() for p in pdf_paths
    )
)

# 문서의 개수 출력
print(len(docs))

In [ ]:
# 문서의 내용 출력 (문서 수가 50개 미만일 수 있으므로 인덱스를 보정)
idx = min(50, len(docs) - 1)
print(docs[idx].page_content[:300])

In [ ]:
# metadata 출력
print(docs[idx].metadata)

## PDFPlumber

PyMuPDF 와 마찬가지로, PDF 와 페이지에 대한 자세한 메타데이터를 얻을 수 있으며 페이지 당 하나의 문서를 반환합니다. 표 추출(`extract_tables`) 기능이 강력합니다.

In [ ]:
# 설치
# !pip install -qU pdfplumber

In [ ]:
import pdfplumber


def pdfplumber_pages(path: Path):
    with pdfplumber.open(path) as pdf:
        doc_info = {k: str(v) for k, v in (pdf.metadata or {}).items()}
        total_pages = len(pdf.pages)
        for page in pdf.pages:
            yield page.extract_text() or "", {
                "total_pages": total_pages,
                "num_tables": len(page.find_tables()),  # 페이지 내 표 개수
                **doc_info,
            }


loader = PagePDFLoader(FILE_PATH, pdfplumber_pages, "pdfplumber")

# 문서 로딩
docs = loader.load()

# 11번째 페이지 데이터 접근
print(docs[10].page_content[:300])

In [ ]:
show_metadata(docs)

## (추가) Docling

IBM 의 오픈소스 문서 변환기 [Docling](https://docling-project.github.io/docling/) 의 LangChain 통합(`langchain-docling`)입니다. 로컬에서 레이아웃·표 구조를 분석하며, 공식 문서의 로더 인터페이스 예제로도 사용됩니다.

- `ExportType.DOC_CHUNKS`(기본): Docling 의 청커로 나눈 조각들을 반환
- `ExportType.MARKDOWN`: 문서 전체를 Markdown 1개로 반환

In [ ]:
# 설치
# !pip install -qU langchain-docling

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

loader = DoclingLoader(file_path=FILE_PATH, export_type=ExportType.MARKDOWN)
docs = loader.load()
print(docs[0].page_content[:500])